# Exercises — Drawdown and the Calmar ratio

[DataCamp exercise](https://campus.datacamp.com/courses/financial-trading-in-python/performance-evaluation-4?ex=5) · see `Notes.md` in this folder for the summary.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Locate the project's data folder regardless of where this notebook runs from
DATA = next(p / "course materials" / "data"
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "course materials" / "data").is_dir())

def load(name):
    """Load an OHLCV CSV with a parsed DatetimeIndex."""
    return pd.read_csv(DATA / name, index_col="Date", parse_dates=True)

def price(name, col, year=None):
    """Single-asset price DataFrame (column = `col`) for use with bt."""
    df = load(name)
    if year:
        df = df[df.index.year == year]
    return df["Close"].rename(col).to_frame()


In [ ]:
import bt
import talib

data = price("AMZN-stock-data.csv", "AMZN")
sma = talib.SMA(data["AMZN"], timeperiod=50)
signal = pd.DataFrame(data["AMZN"].values > sma.values, index=data.index, columns=["AMZN"])
strat = bt.Strategy("SMA50", [bt.algos.SelectWhere(signal),
                              bt.algos.WeighEqually(), bt.algos.Rebalance()])
bt_result = bt.run(bt.Backtest(strat, data))

resInfo = bt_result.stats
print("Max drawdown: {:.2%}".format(resInfo.loc["max_drawdown"].iloc[0]))
print("CAGR:         {:.2%}".format(resInfo.loc["cagr"].iloc[0]))
print("Calmar ratio: {:.2f}".format(resInfo.loc["calmar"].iloc[0]))

### Underwater (drawdown) plot

In [ ]:
eq = bt_result.prices.iloc[:, 0]
dd = eq / eq.cummax() - 1
dd.plot(title="Drawdown (underwater plot)", color="red")
plt.ylabel("Drawdown")
plt.show()

---

## Coding exercises in this section

The `script.py` starter for each Coding exercise under the **Drawdown** video (course exercises 6, 7). Blanks (`____`) are as shown in DataCamp — fill them in to solve.

### 6. Review performance with drawdowns

You backtested a two-MA signal strategy on Tesla data (2019–2020). Obtain all backtest stats in `resInfo`, then get the average drawdown and the average drawdown days.

**Instructions** — see the description above.

In [ ]:
# Obtain all backtest stats
resInfo = ____

# Get the average drawdown
avg_drawdown = ____
print('Average drawdown: %.2f'% avg_drawdown)

# Get the average drawdown days
avg_drawdown_days = ____
print('Average drawdown days: %.0f'% avg_drawdown_days)


### 7. Calculate and review the Calmar ratio

Extract the CAGR and the maximum drawdown from `resInfo`, compute the Calmar ratio manually (CAGR / max drawdown × -1), then retrieve the Calmar ratio directly and compare.

**Instructions** — see the description above.

In [ ]:
# Get the CAGR
cagr = ____
print('Compound annual growth rate: %.4f'% cagr)

# Get the max drawdown
max_drawdown = ____
print('Maximum drawdown: %.2f'% max_drawdown)

# Calculate Calmar ratio manually
calmar_calc = ____ / ____ * (-1)
print('Calmar Ratio calculated: %.2f'% calmar_calc)

# Get the Calmar ratio
calmar = ____
print('Calmar Ratio: %.2f'% calmar)
